# ☕ Mybrew Real-Time Streaming Simulator

Simulates live event data flowing into the Eventhouse KQL DB.

**Run Cell 1 (setup) then Cell 2 (stream).** Open the Real-Time Dashboard to watch.

Events generated:
- 💳 POS transactions (customers buying coffee)
- 📱 Ad impressions & clicks (Meta, Google, Amazon campaigns)
- 🌦️ Weather events (London cold = personalization triggers!)
- 📊 ROI metrics (aggregated campaign performance)

In [ ]:
# Cell 1: Setup - uses mssparkutils for token (built into Fabric)
import random
import time
import json
import requests
from datetime import datetime, timedelta

# KQL DB connection - uses Fabric built-in auth
KUSTO_URI = 'https://trd-t1s1uap7p3xkuvdx9s.z8.kusto.fabric.microsoft.com'
DB_NAME = 'mybrew_eventhouse'

def get_kusto_token():
    """Get token using mssparkutils (built into Fabric notebooks)"""
    return mssparkutils.credentials.getToken('https://kusto.kusto.windows.net')

def run_kql_command(cmd):
    """Execute a KQL management command via REST"""
    token = get_kusto_token()
    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    body = {'db': DB_NAME, 'csl': cmd}
    r = requests.post(f'{KUSTO_URI}/v1/rest/mgmt', headers=headers, json=body)
    if r.status_code != 200:
        print(f'  KQL Error: {r.text[:200]}')
    return r.status_code == 200

def run_kql_query(query):
    """Execute a KQL query via REST"""
    token = get_kusto_token()
    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    body = {'db': DB_NAME, 'csl': query}
    r = requests.post(f'{KUSTO_URI}/v1/rest/query', headers=headers, json=body)
    if r.status_code == 200:
        return r.json()
    return None

# ============ MYBREW DATA DIMENSIONS ============
CUSTOMERS = [
    'C-000001','C-000002','C-000003','C-000004','C-000005',
    'C-000006','C-000007','C-000008','C-000009','C-000010',
    'C-000011','C-000012','C-000015','C-000018','C-000020'
]

STORES = {
    'London': ['STR-LDN-001','STR-LDN-002','STR-LDN-003','STR-LDN-004','STR-LDN-005'],
    'Manchester': ['STR-MAN-001','STR-MAN-002'],
    'Los Angeles': ['STR-LA-001','STR-LA-002'],
    'San Francisco': ['STR-SF-001'],
    'New York': ['STR-NYC-001','STR-NYC-002'],
    'Seattle': ['STR-SEA-001']
}

PRODUCTS_HOT = [('PRD-001','Hazelnut Flat White',4.75), ('PRD-005','Chai Tea Latte',4.50),
                ('PRD-006','Toffee Nut Crunch',5.50), ('PRD-008','Matcha Oat Latte',5.25),
                ('PRD-010','Gingerbread Latte',5.50), ('PRD-011','Americano',3.50)]
PRODUCTS_COLD = [('PRD-002','Iced Caramel Macchiato',5.25), ('PRD-004','Mango Dragonfruit',4.95),
                 ('PRD-007','Cold Brew Nitro',5.00), ('PRD-009','Vanilla Sweet Cream',5.25)]

CAMPAIGNS = {
    'CMP-WW-2026': {'name':'Winter Warmth','publisher':'Meta','cpa_base':21.14},
    'CMP-WW-2026-G': {'name':'Winter Warmth','publisher':'Google','cpa_base':32.00},
    'CMP-WW-2026-A': {'name':'Winter Warmth','publisher':'Amazon','cpa_base':26.92},
    'CMP-SS-2026': {'name':'Summer Sips','publisher':'Meta','cpa_base':18.50},
    'CMP-WB-2026': {'name':'Win-Back Q3','publisher':'Direct','cpa_base':5.00},
    'CMP-TN-2026': {'name':'Tier Nudge','publisher':'Direct','cpa_base':3.50}
}

CREATIVES = {'Meta':'CR-WW-Stories-01','Google':'CR-WW-Display-01',
             'Amazon':'CR-WW-Sponsored-01','Direct':'CR-WB-Push-01'}

# Weather simulation - London is COLD (drives personalization!)
WEATHER_CITIES = [
    {'city':'London',       'temp_base':3,  'condition':'Snow/Sleet', 'precip_base':12.0},
    {'city':'London',       'temp_base':4,  'condition':'Rain',       'precip_base':8.5},
    {'city':'Manchester',   'temp_base':4,  'condition':'Rain',       'precip_base':9.8},
    {'city':'Manchester',   'temp_base':5,  'condition':'Sleet',      'precip_base':7.2},
    {'city':'Los Angeles',  'temp_base':28, 'condition':'Sunny',      'precip_base':0.0},
    {'city':'San Francisco','temp_base':18, 'condition':'Fog',        'precip_base':0.0},
    {'city':'New York',     'temp_base':20, 'condition':'Cloudy',     'precip_base':2.0},
    {'city':'Seattle',      'temp_base':17, 'condition':'Overcast',   'precip_base':0.8}
]

# ============ EVENT GENERATORS ============
def gen_pos_event():
    """Generate a POS transaction event"""
    city = random.choices(
        ['London','Manchester','Los Angeles','San Francisco','New York','Seattle'],
        weights=[50, 10, 15, 8, 12, 5]  # London-heavy for demo
    )[0]
    store = random.choice(STORES[city])
    
    # Cold cities buy hot drinks!
    weather_info = next((w for w in WEATHER_CITIES if w['city'] == city), WEATHER_CITIES[0])
    if weather_info['temp_base'] < 10:
        pid, pname, price = random.choice(PRODUCTS_HOT)
    else:
        pid, pname, price = random.choice(PRODUCTS_HOT + PRODUCTS_COLD)
    
    # 70% of transactions linked to a campaign
    campaign = random.choice(list(CAMPAIGNS.keys())) if random.random() < 0.7 else ''
    amount = round(price + random.uniform(-0.20, 1.50), 2)
    ts = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
    
    return f'TXN-{int(time.time()*1000)}{random.randint(10,99)},{random.choice(CUSTOMERS)},{store},{campaign},{pid},{amount},{ts}'

def gen_impression_event():
    """Generate an ad impression event"""
    cmp_id = random.choice(list(CAMPAIGNS.keys()))
    cmp = CAMPAIGNS[cmp_id]
    pub = cmp['publisher']
    
    # CTR varies by publisher (Meta > Amazon > Google for this demo)
    ctr = {'Meta': 0.18, 'Google': 0.07, 'Amazon': 0.12, 'Direct': 0.40}[pub]
    clicked = random.random() < ctr
    converted = clicked and random.random() < 0.14
    
    spend = round(random.uniform(0.15, 2.50), 2) if pub != 'Direct' else round(random.uniform(0.01, 0.05), 2)
    ts = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
    
    return f'IMP-{int(time.time()*1000)}{random.randint(10,99)},{cmp_id},{random.choice(CUSTOMERS)},{pub},{CREATIVES[pub]},{clicked},{converted},{spend},{ts}'

def gen_weather_event():
    """Generate a weather event (simulated feed)"""
    w = random.choice(WEATHER_CITIES)
    temp = round(w['temp_base'] + random.uniform(-2.0, 2.0), 1)
    precip = round(max(0, w['precip_base'] + random.uniform(-2, 4)), 1)
    ts = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
    return f"{w['city']},{temp},{w['condition']},{precip},{ts}", temp < 5

def gen_roi_event():
    """Generate aggregated ROI metric"""
    cmp_id = random.choice(list(CAMPAIGNS.keys()))
    cmp = CAMPAIGNS[cmp_id]
    pub = cmp['publisher']
    regions = ['London','Manchester','West Coast','East Coast']
    region = random.choice(regions)
    
    # Simulate realistic metrics
    spend = round(random.uniform(5000, 50000), 2)
    roas = {'Meta': 4.2, 'Google': 2.8, 'Amazon': 3.4, 'Direct': 4.9}[pub]
    roas_var = roas + random.uniform(-0.5, 0.5)
    revenue = round(spend * roas_var, 2)
    conversions = int(spend / cmp['cpa_base'] * random.uniform(0.8, 1.2))
    ts = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
    
    return f'{cmp_id},{pub},{region},{revenue},{spend},{round(roas_var,2)},{conversions},{ts}'

# Test connection
print('Testing KQL connection...')
if run_kql_command('.show tables | count'):
    print('\u2705 Connected to KQL DB successfully!')
    print(f'   Cluster: {KUSTO_URI}')
    print(f'   Database: {DB_NAME}')
else:
    print('\u274c Connection failed. Check the Eventhouse is running.')

In [ ]:
# Cell 2: 🚀 START STREAMING (30 minutes, every 5 seconds)
# This is the main simulation loop - run this then open the RT Dashboard!

DURATION_SECONDS = 1800  # 30 minutes
INTERVAL_SECONDS = 5    # batch every 5 seconds

print('═' * 70)
print('  ☕ MYBREW REAL-TIME STREAMING SIMULATOR')
print('  ───────────────────────────────────')
print(f'  Duration: {DURATION_SECONDS}s | Push interval: {INTERVAL_SECONDS}s')
print(f'  Target: mybrew_eventhouse KQL DB')
print(f'  Events: POS + Impressions + Weather + ROI')
print('  ')
print('  👉 Open "Mybrew Real-Time ROI Dashboard" to watch live!')
print('═' * 70)
print()

start_time = time.time()
batch_count = 0
cold_trigger_count = 0
totals = {'pos': 0, 'impressions': 0, 'weather': 0, 'roi': 0,
          'revenue': 0.0, 'ad_spend': 0.0}

while time.time() - start_time < DURATION_SECONDS:
    batch_count += 1
    now_str = datetime.utcnow().strftime('%H:%M:%S')
    
    # Generate batch of events
    n_pos = random.randint(4, 10)
    n_imp = random.randint(12, 25)
    n_weather = random.randint(1, 3)
    
    pos_events = [gen_pos_event() for _ in range(n_pos)]
    imp_events = [gen_impression_event() for _ in range(n_imp)]
    weather_results = [gen_weather_event() for _ in range(n_weather)]
    weather_events = [w[0] for w in weather_results]
    cold_triggers = sum(1 for w in weather_results if w[1])
    cold_trigger_count += cold_triggers
    
    # Every 3rd batch, also push a ROI metric update
    roi_events = [gen_roi_event()] if batch_count % 3 == 0 else []
    
    # Push to KQL DB
    if pos_events:
        cmd = '.ingest inline into table POSTransactions <|\n' + '\n'.join(pos_events)
        run_kql_command(cmd)
    
    if imp_events:
        cmd = '.ingest inline into table AdImpressions <|\n' + '\n'.join(imp_events)
        run_kql_command(cmd)
    
    if weather_events:
        cmd = '.ingest inline into table WeatherEvents <|\n' + '\n'.join(weather_events)
        run_kql_command(cmd)
    
    if roi_events:
        cmd = '.ingest inline into table ROIMetrics <|\n' + '\n'.join(roi_events)
        run_kql_command(cmd)
    
    # Track totals
    totals['pos'] += n_pos
    totals['impressions'] += n_imp
    totals['weather'] += n_weather
    totals['roi'] += len(roi_events)
    batch_rev = sum(float(e.split(',')[5]) for e in pos_events)
    batch_spend = sum(float(e.split(',')[7]) for e in imp_events)
    totals['revenue'] += batch_rev
    totals['ad_spend'] += batch_spend
    
    # Print batch summary
    trigger_str = f' | ❄️ COLD x{cold_triggers}!' if cold_triggers else ''
    roi_str = f' | 📊 ROI' if roi_events else ''
    print(f'[{now_str}] Batch {batch_count:3d} | '
          f'💳 {n_pos} POS (£{batch_rev:.0f}) | '
          f'📱 {n_imp} ads (£{batch_spend:.1f}) | '
          f'🌦️ {n_weather} wx'
          f'{trigger_str}{roi_str}')
    
    time.sleep(INTERVAL_SECONDS)

# Summary
print()
print('═' * 70)
print('  ✅ SIMULATION COMPLETE')
print(f'  Duration: {DURATION_SECONDS}s | Batches: {batch_count}')
print(f'  ───────────────────────────────────')
print(f'  💳 POS Transactions:  {totals["pos"]}')
print(f'  📱 Ad Impressions:    {totals["impressions"]}')
print(f'  🌦️ Weather Events:    {totals["weather"]}')
print(f'  📊 ROI Updates:       {totals["roi"]}')
print(f'  ───────────────────────────────────')
print(f'  Total Revenue:  £{totals["revenue"]:,.2f}')
print(f'  Total Ad Spend: £{totals["ad_spend"]:,.2f}')
print(f'  Simulated ROAS: {totals["revenue"]/max(totals["ad_spend"],1):.2f}x')
print(f'  ❄️ Cold Triggers:  {cold_trigger_count} (personalization fired!)')
print('═' * 70)

In [ ]:
# Cell 3: Verify results - query KQL tables
print('\n📊 KQL Table Counts After Streaming:')
print('-' * 40)
tables = ['POSTransactions', 'AdImpressions', 'WeatherEvents', 'ROIMetrics']
for t in tables:
    result = run_kql_query(f'{t} | count')
    if result:
        count = result['Tables'][0]['Rows'][0][0]
        print(f'  {t}: {count} rows')

print('\n\n📈 Live ROI by Publisher:')
print('-' * 50)
result = run_kql_query("""
AdImpressions
| summarize Imp=count(), Clicks=countif(clicked), Conv=countif(converted), 
            Spend=round(sum(spend_amount_gbp),2) by publisher
| extend CTR_pct=round(100.0*todouble(Clicks)/todouble(Imp), 1),
         CPA=round(Spend/todouble(max_of(Conv,1)), 2)
| order by Imp desc
""")
if result:
    print(f'{"Publisher":<10}{"Impressions":<13}{"Clicks":<8}{"CTR%":<7}{"Conv":<7}{"Spend":<12}{"CPA"}')
    for row in result['Tables'][0]['Rows']:
        print(f'{row[0]:<10}{row[1]:<13}{row[2]:<8}{row[4]:<7}{row[3]:<7}\u00a3{row[5]:<10}\u00a3{row[6]}')

print('\n\n❄️ Cold Weather Triggers (Personalization):')
print('-' * 50)
result = run_kql_query("""
WeatherEvents 
| where temp_c < 5 
| summarize ColdEvents=count(), AvgTemp=round(avg(temp_c),1) by city 
| order by ColdEvents desc
""")
if result:
    for row in result['Tables'][0]['Rows']:
        print(f'  {row[0]}: {row[1]} cold events (avg {row[2]}\u00b0C) \u2192 Hazelnut Flat White pushed!')